# 🧠 RMT-LLM Research: бесплатный GPU-сервер для TinyGPT v3

Дообучение модели **TinyGPT v3** из репозитория
[wild8highlander/rmt-llm-research](https://github.com/wild8highlander/rmt-llm-research)
на корпусе формул RMT — на бесплатном GPU Google Colab.

**Что делает ноутбук:** клонирует репозиторий, собирает корпус формул (277 вычислительных пар),
обучает конфигурацию `config_train_4m` (~4.15M параметров, RoPE+GQA, 150 эпох — цель ROADMAP:
match_rate ≥ 25%), измеряет эффективность (loss, перплексия, match_rate по эпохам) и строит графики.

**Runtime:** `Runtime → Change runtime type → T4 GPU` (бесплатно).
Можно запустить и на CPU, но ~в 50 раз медленнее.

In [ ]:
# 1. Клонируем репозиторий и настраиваем пути
import os, sys, subprocess
if not os.path.exists('rmt-llm-research'):
    subprocess.run(['git', 'clone', 'https://github.com/wild8highlander/rmt-llm-research.git'],
                   check=True)
REPO = '/content/rmt-llm-research'
LAB = os.path.join(REPO, 'laboratory', 'python', 'lab_en')
SRC = os.path.join(REPO, 'src', 'rmt_llm')
sys.path.insert(0, LAB); sys.path.insert(0, SRC)
print('Репозиторий готов:', REPO)

In [ ]:
# 2. Собираем корпус формул (тот же пайплайн, что в веб-приложении)
# Вариант A: использовать corpus_builder_v3.py из репозитория (код+доки проекта, до 20 МБ)
from corpus_builder_v3 import CorpusBuilder, CorpusConfig
builder = CorpusBuilder(CorpusConfig(max_bytes=2_000_000))
corpus_bytes = builder.build(roots=[REPO])
print(f'Корпус: {len(corpus_bytes):,} байт')
print(builder.summary())

# Вариант B (рекомендуется для формул): числовые пары от всех функций ядра
# Скопируйте scripts/build_corpus.py из веб-приложения — он генерирует 277 пар
# вида "mp_bounds(q=0.5, sigma2=1.0) => lambda_minus = 0.0858, lambda_plus = 2.9142" 

In [ ]:
# 3. Токенизация BPE-512
import numpy as np
from tiny_gpt_trainer import BPETokenizer
tok = BPETokenizer(vocab_size=512)
tok.train(corpus_bytes)
ids = tok.encode_bytes(corpus_bytes)
n_val = max(len(ids)//10, 64)
train_ids, val_ids = ids[:-n_val], ids[-n_val:]
print(f'Токены: train={len(train_ids):,}, val={len(val_ids):,}')

In [ ]:
# 4. Модель «из коробки»: TinyGPT v3 config_train_4m (4.15M параметров)
from tiny_gpt_v3 import TinyGPTV3, TinyGPTV3Config, config_train_4m
from tiny_gpt_trainer_v3 import TrainerV3, TrainConfig

cfg = config_train_4m()           # 10 слоёв, hidden=192, RoPE, GQA 6/2
cfg.weight_tying = True; cfg.label_smoothing = 0.1
cfg.grad_clip_norm = 1.0; cfg.dropout = 0.1
model = TinyGPTV3(cfg)
print(f'Параметров: {cfg.params_count:,}')

train_cfg = TrainConfig(
    epochs=150, batch_size=8, grad_accum_steps=2,
    max_lr=3e-3, warmup_ratio=0.1, min_lr_ratio=0.1,
    eval_interval=5, early_stopping_patience=0, verbose=True)
trainer = TrainerV3(model, train_cfg)

In [ ]:
# 5. Обучение (на T4 ~3-5 мин/эпоха; цель ROADMAP: val match_rate >= 25%)
history = trainer.train(train_ids, val_ids=val_ids)
model.save_weights('tiny_gpt_4m_formula.npz')
tok.save('tiny_gpt_4m_bpe.json')
import json
json.dump(history, open('training_history_4m.json', 'w'), indent=1)

In [ ]:
# 6. Измерение эффективности: сколько модель становится лучше по эпохам
import matplotlib.pyplot as plt
L, VL = history['losses'], history['val_losses']
MR = history['val_match_rates']
base_loss = np.exp(np.log(L[0]))  # первая эпоха ~ случайный уровень
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(L, label='train'); ax[1].plot(VL, label='val', color='g')
ax[0].set_title('Loss'); ax[1].set_title('Val loss')
ax[0].legend(); ax[1].legend()
ax[2].plot(np.array(MR)*100, color='r')
ax[2].axhline(25, ls='--', color='gray'); ax[2].set_title('Val match rate, % (цель 25%)')
for a in ax: a.set_xlabel('эпоха')
plt.tight_layout(); plt.savefig('rmt_training_metrics.png', dpi=150)
print(f'Прирост val loss: {VL[0]:.2f} -> {VL[-1]:.2f} (x{VL[0]/max(VL[-1],1e-9):.2f} лучше)')
print(f'Match rate: {MR[0]:.1%} -> {MR[-1]:.1%}')

In [ ]:
# 7. RMT-диагностика: спектр скрытых состояний обученной модели
# BBP-тест: выходит ли lambda_max за верхнюю границу Марченко-Пастура?
import math
prompt_ids = tok.encode('mp_bounds(q=0.5, sigma2=1.0)')[:64]
out = model.generate(np.array(prompt_ids), max_new_tokens=24, temperature=0.0, capture_hidden=True)
for li, h in enumerate(out['hidden_snapshots'][0]):
    cov = np.cov(h.T); ev = np.linalg.eigvalsh(cov); ev = ev[ev > 1e-10]
    T, H = h.shape; q = H/T; s2 = ev.mean()
    mp_up = s2*(1+math.sqrt(min(q,1)))**2
    print(f'слой {li}: lambda_max={ev.max():.3f} vs MP_up={mp_up:.3f} ->', 'SPIKE' if ev.max() > mp_up*1.05 else 'bulk')
print('Сгенерировано:', tok.decode(out['output_ids']))

## Что дальше
- **Скачайте** `tiny_gpt_4m_formula.npz` + `training_history_4m.json` — их можно отдать в веб-приложение
  (положить в папку `model/`) или в репозиторий как release-артефакт.
- **Сравните** с локальной моделью 447K: цель ROADMAP — match_rate ≥ 25% на 4M/12M конфигах.
- Для config_12m (12.5M параметров) увеличьте `epochs=150`, `batch_size=8` — уместится в T4.
- Открытые вопросы ROADMAP проверьте `scripts/open_questions.py` из веб-приложения (работает и здесь).